In [1]:
import pandas as pd

file_path = "../RawData/NewsData10200records.csv"
data = pd.read_csv(file_path)


In [2]:
print("\nข้อมูลสรุป:")
print(data.info()) 


ข้อมูลสรุป:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12020 entries, 0 to 12019
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Public_Date_Time   12020 non-null  object
 1   URL                12020 non-null  object
 2   Title              12020 non-null  object
 3   Body               12020 non-null  object
 4   Verify_Department  9950 non-null   object
 5   Types              12017 non-null  object
 6   category           12020 non-null  object
 7   Viewers            12020 non-null  int64 
 8   Hashtag            12020 non-null  object
dtypes: int64(1), object(8)
memory usage: 845.3+ KB
None


In [3]:
print("\nจำนวนข้อมูลที่หายไปในแต่ละคอลัมน์:")
print(data.isnull().sum())


จำนวนข้อมูลที่หายไปในแต่ละคอลัมน์:
Public_Date_Time        0
URL                     0
Title                   0
Body                    0
Verify_Department    2070
Types                   3
category                0
Viewers                 0
Hashtag                 0
dtype: int64


In [4]:
print("\nสถิติพื้นฐาน:")
print(data.describe())


สถิติพื้นฐาน:
             Viewers
count   12020.000000
mean     1664.520383
std      7424.509748
min         2.000000
25%       150.000000
50%       352.000000
75%      1381.750000
max    462320.000000


In [5]:
print("\nค่าที่ไม่ซ้ำในคอลัมน์ 'Types':")
print(data['Types'].unique())


ค่าที่ไม่ซ้ำในคอลัมน์ 'Types':
['ข่าวจริง' 'ข่าวปลอม' 'ข่าวบิดเบือน' 'คลังความรู้' 'อาชญากรรมออนไลน์'
 'ข่าวอื่นๆ' 'กิจกรรม' 'นโยบายรัฐบาล-ข่าวสาร' 'ข่าวสาร' nan 'การเงิน-หุ้น'
 'ผลิตภัณฑ์สุขภาพ' 'ยาเสพติด']


In [7]:
# ลบคอลัมน์ 'Verify_Department'
data = data.drop(columns=['Verify_Department','Public_Date_Time', 'URL'])


In [6]:
data_clean = data.dropna()

In [7]:
print(data_clean.info())

<class 'pandas.core.frame.DataFrame'>
Index: 9950 entries, 0 to 11920
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Public_Date_Time   9950 non-null   object
 1   URL                9950 non-null   object
 2   Title              9950 non-null   object
 3   Body               9950 non-null   object
 4   Verify_Department  9950 non-null   object
 5   Types              9950 non-null   object
 6   category           9950 non-null   object
 7   Viewers            9950 non-null   int64 
 8   Hashtag            9950 non-null   object
dtypes: int64(1), object(8)
memory usage: 777.3+ KB
None


In [8]:
data_clean.loc[:, 'Types'] = data_clean['Types'].astype('category')
print(data_clean['Types'].isna().sum())  # ตรวจสอบจำนวน NaN
print(data_clean['Types'].unique()) 

0
['ข่าวจริง' 'ข่าวปลอม' 'ข่าวบิดเบือน' 'คลังความรู้' 'อาชญากรรมออนไลน์'
 'ข่าวอื่นๆ' 'กิจกรรม' 'นโยบายรัฐบาล-ข่าวสาร' 'ข่าวสาร' 'ผลิตภัณฑ์สุขภาพ']


In [9]:
data_clean.loc[:, 'Types'] = data_clean['Types'].str.strip()

In [10]:
# กรองข้อมูลให้เหลือแค่ 'ข่าวจริง' และ 'ข่าวปลอม' เท่านั้น
df_filtered = data_clean[data_clean['Types'].isin(['ข่าวจริง', 'ข่าวปลอม'])]

# ตรวจสอบจำนวนของแต่ละประเภทหลังการกรอง
print("\nจำนวนของแต่ละประเภทในคอลัมน์ 'Types' หลังการกรอง:")
print(df_filtered['Types'].value_counts())


จำนวนของแต่ละประเภทในคอลัมน์ 'Types' หลังการกรอง:
Types
ข่าวปลอม    5721
ข่าวจริง    1917
Name: count, dtype: int64


In [11]:
# ดุลข่าวแต่ละประเภท ลดข่าวปลอมลง
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

print("จำนวนข้อมูลในแต่ละคลาสก่อนทำ Oversampling/Undersampling:")
print(Counter(df_filtered['Types']))

X = df_filtered.drop(columns=['Types'])  # Features
y = df_filtered['Types']  # Target

undersample = RandomUnderSampler(sampling_strategy={'ข่าวปลอม': 2000}, random_state=42)
X_under, y_under = undersample.fit_resample(X, y)

print("จำนวนข้อมูลหลังทำ Undersampling:", Counter(y_under))

จำนวนข้อมูลในแต่ละคลาสก่อนทำ Oversampling/Undersampling:
Counter({'ข่าวปลอม': 5721, 'ข่าวจริง': 1917})
จำนวนข้อมูลหลังทำ Undersampling: Counter({'ข่าวปลอม': 2000, 'ข่าวจริง': 1917})


In [12]:
print(X_under.info())  # ดู 5 แถวแรกของ features
print(pd.Series(y_under).value_counts())

<class 'pandas.core.frame.DataFrame'>
Index: 3917 entries, 0 to 731
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Public_Date_Time   3917 non-null   object
 1   URL                3917 non-null   object
 2   Title              3917 non-null   object
 3   Body               3917 non-null   object
 4   Verify_Department  3917 non-null   object
 5   category           3917 non-null   object
 6   Viewers            3917 non-null   int64 
 7   Hashtag            3917 non-null   object
dtypes: int64(1), object(7)
memory usage: 275.4+ KB
None
Types
ข่าวปลอม    2000
ข่าวจริง    1917
Name: count, dtype: int64


In [15]:
print(X_under.info())
print(y_under.count())

<class 'pandas.core.frame.DataFrame'>
Index: 4245 entries, 0 to 1875
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Title     4245 non-null   object
 1   Body      4245 non-null   object
 2   category  4245 non-null   object
 3   Viewers   4245 non-null   int64 
 4   Hashtag   4245 non-null   object
dtypes: int64(1), object(4)
memory usage: 199.0+ KB
None
4245


In [16]:
print(y_under)

0       ข่าวจริง
2       ข่าวจริง
4       ข่าวจริง
6       ข่าวจริง
11      ข่าวจริง
          ...   
4205    ข่าวปลอม
2126    ข่าวปลอม
2810    ข่าวปลอม
4256    ข่าวปลอม
1875    ข่าวปลอม
Name: Types, Length: 4245, dtype: object


In [13]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_under = label_encoder.fit_transform(y_under)

# แสดงผลลัพธ์ที่แปลง 0 1 แล้ว
print(y_under)

[0 0 0 ... 1 1 1]


Save Result

In [ ]:
import joblib
# บันทึก y_under
joblib.dump(X_under, 'result/X_under.pkl')
joblib.dump(y_under, 'result/y_under.pkl')

print("✅ บันทึกข้อมูลเรียบร้อยแล้ว!")


✅ บันทึกข้อมูลเรียบร้อยแล้ว!
